# EDA Relevansi Sumber Data

`docs/data-lineage.md` sudah mencatat **apa** yang nyata dan apa yang sintesis.
Notebook ini menjawab pertanyaan berikutnya, yang tidak dijawab dokumen itu:
**seberapa relevan sinyal nyata tersebut terhadap segmen yang hendak dimodelkan** —
debitur komersial Indonesia berpenjualan Rp 30–300 miliar.

Bedanya penting:

> Kualitas data menentukan apakah temuan Anda **nyata**.
> Relevansi data menentukan apakah temuan itu **berlaku di tempat Anda hendak memakainya**.
> Proyek ini bisa membuktikan yang pertama, dan hanya bisa *mengukur batas* yang kedua.

---

## Yang diukur

| § | Pertanyaan | Cara |
|---|---|---|
| 1 | Populasi sumber seperti apa? | ukuran perusahaan vs pita segmen |
| 2 | Apakah base rate-nya sepadan? | tingkat gagal bayar sumber vs ABT |
| 3 | **Apakah sinyalnya selamat sampai gold?** | AUC univariat rasio→label, sumber vs `abt_pd` |
| 4 | Apakah struktur antar-rasionya selamat? | matriks korelasi, sumber vs gold |
| 5 | Bagaimana dengan LGD dan topologi graf? | transfer SBA→portofolio, sebaran derajat |
| 6 | Ringkasan | tabel skor relevansi per sumber |

§3 dan §4 adalah intinya. Kalau relasi di sumber tidak muncul lagi di `abt_pd`,
join sintetisnya merusak sinyal — itu cacat pipeline yang harus diperbaiki.
Kalau muncul utuh, sinyalnya selamat, dan sisanya murni soal relevansi populasi —
batas yang harus dinyatakan, bukan diperbaiki.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

AKAR = Path.cwd()
while not (AKAR / "pipelines").is_dir() and AKAR != AKAR.parent:
    AKAR = AKAR.parent
sys.path.insert(0, str(AKAR))

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

def bronze(n): return pd.read_parquet(AKAR / "data" / "bronze" / f"{n}.parquet")
def silver(n): return pd.read_parquet(AKAR / "data" / "silver" / f"{n}.parquet")
def gold(n):   return pd.read_parquet(AKAR / "data" / "gold" / f"{n}.parquet")

SEGMEN_MIN, SEGMEN_MAKS = 30e9, 300e9
print("akar proyek:", AKAR)

## 0 · Peta sumber

Tujuh berkas publik, masing-masing menyumbang satu jenis sinyal. Kolom
**populasi asli** menyebut siapa yang sebenarnya diukur di sana — dan di situlah
seluruh persoalan relevansi bermula.

In [ ]:
SUMBER = pd.DataFrame([
    dict(berkas="american_bankruptcy.csv", tabel="br_us_panel",
         peran="rasio keuangan + label gagal bayar (tulang punggung)",
         populasi="emiten AS (NYSE/NASDAQ), panel tahunan"),
    dict(berkas="data.csv (Taiwan)", tabel="br_taiwan_ratio",
         peran="rasio tambahan, dicocokkan pada label + kuintil DER/ROA",
         populasi="perusahaan tercatat Taiwan"),
    dict(berkas="SBAnational.csv", tabel="br_sba",
         peran="LGD, tenor, porsi penjaminan, kelengkapan dokumen",
         populasi="pinjaman UMKM terjamin SBA di AS"),
    dict(berkas="corporate_rating.csv", tabel="br_rating",
         peran="sebaran rating & rasio per sektor",
         populasi="emiten AS berperingkat agensi"),
    dict(berkas="LI-Small_Trans.csv", tabel="br_aml_transfer",
         peran="topologi transfer: siklus, fan-in/out",
         populasi="transaksi sintetis IBM (bukan bank nyata)"),
    dict(berkas="ICIJ Offshore Leaks", tabel="br_icij_entity",
         peran="kepemilikan berlapis, rangkap jabatan, alamat bersama",
         populasi="entitas lepas pantai dunia"),
])
SUMBER["baris"] = [f"{len(bronze(t)):,}".replace(",", ".") for t in SUMBER["tabel"]]
SUMBER[["berkas", "peran", "populasi", "baris"]]

**Tidak satu pun populasi di atas adalah debitur komersial Indonesia.** Itu bukan
cacat yang bisa diperbaiki — tidak ada dataset publik berisi debitur bank
Indonesia beserta outcome-nya. Yang bisa dilakukan hanyalah mengukur jaraknya,
lalu menyatakannya secara eksplisit di laporan.

## 1 · Kesenjangan populasi

Tulang punggungnya `american_bankruptcy.csv` — emiten AS. Segmen sasarannya
perusahaan Indonesia berpenjualan Rp 30–300 M, yang di AS tidak akan pernah
tercatat di bursa. Seberapa jauh?

In [ ]:
us = silver("sl_us_panel")
d = gold("abt_pd")

# Panel AS dalam juta USD. Kurs kasar hanya untuk menempatkan ordo besarannya,
# bukan konversi akuntansi.
KURS = 15_500
us_rp = us["total_revenue"] * 1e6 * KURS
us_rp = us_rp[us_rp > 0]

print("Panel AS, penjualan ditempatkan pada skala rupiah:")
for k, v in us_rp.describe(percentiles=[.1, .5, .9]).items():
    if k in ("min", "10%", "50%", "90%", "max"):
        print(f"  {k:>4s}: Rp {v/1e9:>18,.1f} miliar".replace(",", "."))

di_segmen = us_rp.between(SEGMEN_MIN, SEGMEN_MAKS)
print(f"\nBerada di pita Rp 30-300 M : {di_segmen.sum():,} dari {len(us_rp):,} "
      f"({di_segmen.mean()*100:.1f}%)".replace(",", "."))
print(f"Di BAWAH pita              : {(us_rp < SEGMEN_MIN).mean()*100:.1f}%")
print(f"Di ATAS pita               : {(us_rp > SEGMEN_MAKS).mean()*100:.1f}%")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].hist(np.log10(us_rp), bins=60, color="#4C72B0")
for x, lbl in [(np.log10(SEGMEN_MIN), "Rp 30 M"), (np.log10(SEGMEN_MAKS), "Rp 300 M")]:
    ax[0].axvline(x, color="crimson", ls="--", lw=1.2)
    ax[0].text(x, ax[0].get_ylim()[1] * .93, lbl, color="crimson", fontsize=8, ha="center")
ax[0].set_title("SUMBER — penjualan emiten AS")
ax[0].set_xlabel("log10 penjualan (Rp)")

ax[1].hist(np.log10(d["fin_penjualan_rp"]), bins=40, color="#55A868")
ax[1].set_title("GOLD — abt_pd setelah penskalaan peringkat")
ax[1].set_xlabel("log10 penjualan (Rp)")
ax[1].set_xlim(ax[0].get_xlim())
plt.tight_layout(); plt.show()

Sebaran sumber terentang belasan ordo besaran; sebaran gold dijepit ke satu pita
sempit. Pipeline menanganinya lewat **penskalaan berbasis peringkat**
(`skala_rupiah`): urutan besar–kecil dipertahankan, nilainya dipetakan ulang.

Konsekuensinya harus dipahami: yang dibawa lintas transformasi adalah **posisi
relatif**, bukan besaran absolut. Model bisa belajar "perusahaan ini lebih
berisiko dari itu"; ia tidak bisa belajar "perusahaan berpenjualan Rp 50 M punya
PD sekian" — karena tidak ada satu pun perusahaan Rp 50 M sungguhan di sumbernya.

## 2 · Apakah base rate-nya sepadan?

In [ ]:
taiwan = silver("sl_taiwan_ratio")
sba = silver("sl_sba")
ews = gold("abt_ews")

tb = pd.DataFrame([
    dict(populasi="american_bankruptcy (emiten AS)", n=len(us),
         tingkat=us["label_default"].mean(), horizon="sepanjang panel"),
    dict(populasi="Taiwan (emiten tercatat)", n=len(taiwan),
         tingkat=taiwan["label_default_taiwan"].mean(), horizon="kebangkrutan, lintas tahun"),
    dict(populasi="SBA (pinjaman UMKM AS)", n=len(sba),
         tingkat=sba["is_default"].mean(), horizon="charge-off sepanjang umur pinjaman"),
    dict(populasi="abt_pd (hasil)", n=int(d["y_default_12bln"].notna().sum()),
         tingkat=d["y_default_12bln"].mean(), horizon="12 bulan"),
    dict(populasi="abt_ews (hasil)", n=int(ews["y_default_6bln"].notna().sum()),
         tingkat=ews["y_default_6bln"].mean(), horizon="6 bulan"),
])
tb["tingkat"] = (tb["tingkat"] * 100).round(2).astype(str) + "%"
tb

Angka sumber **tidak bisa** dibandingkan langsung dengan hasilnya karena
horizonnya berbeda: 17,5% SBA adalah charge-off sepanjang umur pinjaman
(bertahun-tahun), sedangkan 3,2% `abt_pd` adalah jendela 12 bulan.

Setelah horizonnya disamakan, hasilnya berada pada ordo yang wajar untuk kredit
komersial. **Base rate bukan sumber ketidakrelevanan di sini** — yang bermasalah
adalah populasi dan struktur, bukan tingkat kejadiannya.

## 3 · Apakah sinyalnya selamat sampai gold?  ⟵ inti notebook

Pipeline menempelkan tujuh dataset pada satu `cif` sintetis. Kalau proses itu
merusak relasi rasio→gagal bayar, seluruh pemodelan kehilangan dasarnya.

Ukurannya **AUC univariat** — kemampuan satu rasio membedakan gagal bayar dari
tidak, tanpa melatih model apa pun. Dihitung di sumber, lalu di `abt_pd`, lalu
dibandingkan. Yang dicari bukan angka yang sama persis, melainkan **arah yang
konsisten**.

In [ ]:
def auc_univariat(x: pd.Series, y: pd.Series) -> float:
    '''AUC satu variabel lewat statistik Mann-Whitney - tanpa melatih model.'''
    m = x.notna() & y.notna()
    x, y = x[m], y[m].astype(int)
    if y.nunique() < 2 or len(x) == 0:
        return np.nan
    r = x.rank()
    n1, n0 = int((y == 1).sum()), int((y == 0).sum())
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


PASANGAN = {
    "der": "fin_der",
    "roa": "fin_roa",
    "icr": "fin_icr",
    "current_ratio": "fin_current_ratio",
    "quick_ratio": "fin_quick_ratio",
    "operating_margin": "fin_operating_margin",
    "gross_margin": "fin_gross_margin",
    "wc_to_ta": "fin_wc_to_ta",
    "re_to_ta": "fin_re_to_ta",
    "asset_turnover": "fin_asset_turnover",
    "debt_to_ebitda": "fin_debt_to_ebitda",
    "cfo_to_liability": "fin_cfo_to_liability",
    "growth_penjualan": "fin_growth_penjualan",
}
d_lab = d[d["y_default_12bln"].notna()]
baris = []
for k_us, k_gold in PASANGAN.items():
    if k_us not in us.columns or k_gold not in d_lab.columns:
        continue
    a_src = auc_univariat(us[k_us], us["label_default"])
    a_gld = auc_univariat(d_lab[k_gold], d_lab["y_default_12bln"])
    baris.append(dict(
        rasio=k_us, auc_sumber=a_src, auc_gold=a_gld,
        kekuatan_sumber=abs(a_src - .5), kekuatan_gold=abs(a_gld - .5),
        arah_sama=bool(np.sign(a_src - .5) == np.sign(a_gld - .5)),
    ))
sinyal = pd.DataFrame(baris).sort_values("kekuatan_sumber", ascending=False)
sinyal.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 5.4))
ax.axhline(.5, color="grey", lw=.8)
ax.axvline(.5, color="grey", lw=.8)
ax.plot([.25, .8], [.25, .8], ls="--", color="grey", lw=.9, label="sinyal selamat utuh")
warna = ["#55A868" if s else "#C44E52" for s in sinyal["arah_sama"]]
ax.scatter(sinyal["auc_sumber"], sinyal["auc_gold"], c=warna, s=75, zorder=3)
for _, r in sinyal.iterrows():
    ax.annotate(r["rasio"], (r["auc_sumber"], r["auc_gold"]),
                fontsize=7.5, xytext=(5, 4), textcoords="offset points")
ax.set_xlabel("AUC univariat di SUMBER (panel AS)")
ax.set_ylabel("AUC univariat di GOLD (abt_pd)")
ax.set_title("Apakah relasi rasio→gagal bayar selamat melewati join sintetis?")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

n_sama = int(sinyal["arah_sama"].sum())
print(f"Arah sinyal konsisten : {n_sama} dari {len(sinyal)} rasio")
print(f"Kekuatan rata-rata    : sumber {sinyal.kekuatan_sumber.mean():.3f} -> "
      f"gold {sinyal.kekuatan_gold.mean():.3f} "
      f"({sinyal.kekuatan_gold.mean()/sinyal.kekuatan_sumber.mean()*100:.0f}% bertahan)")
if n_sama < len(sinyal):
    print("\nArah TERBALIK - periksa sebelum dipakai memodelkan:")
    print(sinyal[~sinyal.arah_sama][["rasio", "auc_sumber", "auc_gold"]].to_string(index=False))

**Cara membacanya.** Titik hijau = arah konsisten (rasio yang menandakan risiko di
sumber tetap menandakan risiko di gold). Merah = arah terbalik — itu yang
berbahaya, karena model akan mempelajari hubungan yang bertentangan dengan
ekonomi, dan hasilnya tetap terlihat masuk akal sampai ada yang memeriksa tandanya.

### Hasilnya bukan yang diduga

Arahnya konsisten — bagus. Tapi perhatikan posisi titik-titiknya terhadap garis
diagonal: hampir semuanya **jauh dari 0,5 di sumbu tegak dibanding sumbu datar**.
Sinyalnya tidak melemah melewati pipeline; ia justru menjadi **sekitar dua kali
lebih kuat**.

| rasio | AUC di sumber | AUC di gold |
|---|---|---|
| `debt_to_ebitda` | 0,628 | **0,737** |
| `der` | 0,596 | **0,720** |
| `icr` | 0,398 | **0,269** |
| `roa` | 0,389 | **0,261** |

Penyebabnya ada di `joins.py`: baris Taiwan dicocokkan memakai kunci
`label + kuintil DER + kuintil ROA`. Artinya debitur berlabel sama dipasangkan
dengan rasio yang mirip — dan itu **memperkuat** hubungan rasio↔label melampaui
yang benar-benar ada di data AS asli. Penskalaan peringkat dan pemilihan 6.000
perusahaan dari 78.682 tahun-perusahaan ikut mempertajamnya.

### Kenapa ini penting

Setiap AUC yang Anda laporkan di data ini **optimistis** — bukan karena bocor
(kolom `tw_*` sudah dibuang dari ABT), melainkan karena populasinya dirakit
sedemikian rupa sehingga hubungannya lebih rapi daripada kenyataan. Ini bias yang
terpisah dari, dan menumpuk di atas, kesenjangan relevansi populasi di §1.

Konsekuensi praktisnya: AUC 0,75–0,83 yang didapat di sini **tidak** boleh
dibaca sebagai perkiraan performa di portofolio nyata. Sebutkan di laporan bahwa
kekuatan sinyal di data sintetis ini kira-kira 2× kekuatan di sumber nyatanya.

## 4 · Apakah struktur antar-rasio ikut selamat?

AUC univariat hanya melihat satu rasio pada satu waktu. Tapi model belajar dari
**pola** — DER tinggi yang muncul bersamaan dengan ICR rendah dan margin tipis.
Kalau join sintetis merusak korelasi antar-rasio, model akan mempelajari pola
yang tidak ada di dunia nyata.

Ini juga penjelasan temuan sebelumnya: model meleset 3× saat diberi populasi
dengan fitur yang ditarik independen. Struktur korelasi menentukan level PD.

In [ ]:
RASIO_US = [k for k in PASANGAN if k in us.columns]
RASIO_GOLD = [PASANGAN[k] for k in RASIO_US]

c_src = us[RASIO_US].corr(method="spearman")
c_gld = d[RASIO_GOLD].corr(method="spearman")
c_gld.index = RASIO_US
c_gld.columns = RASIO_US

fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
kw = dict(vmin=-1, vmax=1, cmap="RdBu_r", square=True, cbar=False,
          xticklabels=True, yticklabels=True, annot=False)
sns.heatmap(c_src, ax=ax[0], **kw); ax[0].set_title("SUMBER (panel AS)")
sns.heatmap(c_gld, ax=ax[1], **kw); ax[1].set_title("GOLD (abt_pd)")
selisih = c_gld - c_src
sns.heatmap(selisih, ax=ax[2], vmin=-1, vmax=1, cmap="PuOr", square=True,
            cbar=True, xticklabels=True, yticklabels=True)
ax[2].set_title("SELISIH (gold − sumber)")
for a in ax:
    a.tick_params(labelsize=7)
plt.tight_layout(); plt.show()

segitiga = np.triu(np.ones_like(c_src, dtype=bool), k=1)
v_src, v_gld = c_src.values[segitiga], c_gld.values[segitiga]
print(f"Korelasi antar matriks korelasi : {np.corrcoef(v_src, v_gld)[0,1]:+.3f}")
print(f"Rata-rata |selisih| per pasangan: {np.abs(v_gld - v_src).mean():.3f}")
print(f"Pasangan yang BERBALIK tanda    : {int((np.sign(v_src) != np.sign(v_gld)).sum())} "
      f"dari {len(v_src)}")

In [ ]:
# Pasangan dengan pergeseran terbesar - inilah yang paling mungkin membuat model
# mempelajari pola yang tidak ada di sumbernya.
pasangan = []
for i, a in enumerate(RASIO_US):
    for j, b in enumerate(RASIO_US):
        if i < j:
            pasangan.append(dict(pasangan=f"{a} ~ {b}",
                                 sumber=c_src.iloc[i, j], gold=c_gld.iloc[i, j],
                                 selisih=c_gld.iloc[i, j] - c_src.iloc[i, j]))
geser = pd.DataFrame(pasangan).reindex(
    pd.DataFrame(pasangan)["selisih"].abs().sort_values(ascending=False).index)
geser.head(10).round(3)

## 5 · LGD dan topologi graf

Dua sumber lain punya cara pengukuran relevansi yang berbeda.

**SBA → LGD.** Ini satu-satunya tempat di proyek yang punya uji transfer
luar-domain sungguhan: latih di pinjaman SBA yang tidak dipakai, terapkan ke
portofolio Indonesia. Kalau R² luar-domain runtuh dibanding dalam-domain, itu
ukuran langsung dari ketidakrelevanan.

In [ ]:
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_error

lgd = gold("abt_lgd")
sumber_lgd = gold("abt_lgd_sumber")
latih = sumber_lgd[~sumber_lgd["sba_loan_nr"].isin(set(lgd["src_sba_loannr"]))].copy()

F = ["app_tenor_bulan", "app_jenis_fasilitas", "app_revolving", "app_sektor_kbli",
     "app_skala_pegawai", "app_perusahaan_baru", "app_dokumen_ringkas", "app_porsi_penjaminan"]
terap = lgd.copy()
for c in F:
    if str(latih[c].dtype) in ("object", "string", "bool"):
        kat = pd.api.types.CategoricalDtype(
            sorted(set(latih[c].dropna()) | set(terap[c].dropna())))
        latih[c] = latih[c].astype(kat)
        terap[c] = terap[c].astype(kat)

m = lgb.LGBMRegressor(objective="regression", n_estimators=400, learning_rate=.05,
                      num_leaves=31, min_child_samples=50, verbose=-1,
                      random_state=7).fit(latih[F], latih["y_lgd_realisasi"])
hold = latih.sample(min(5000, len(latih)), random_state=1)
hasil = pd.DataFrame([
    dict(penerapan="DALAM domain (holdout SBA)", n=len(hold),
         MAE=mean_absolute_error(hold["y_lgd_realisasi"], m.predict(hold[F])),
         R2=r2_score(hold["y_lgd_realisasi"], m.predict(hold[F]))),
    dict(penerapan="LUAR domain (portofolio ID)", n=len(terap),
         MAE=mean_absolute_error(terap["y_lgd_realisasi"], m.predict(terap[F])),
         R2=r2_score(terap["y_lgd_realisasi"], m.predict(terap[F]))),
])
print(hasil.round(3).to_string(index=False))

pred = m.predict(terap[F])
cov = terap["app_coverage_ratio"] / terap["app_ead_thd_plafon"]
print(f"\ncorr(prediksi, aktual)   : {np.corrcoef(pred, terap['y_lgd_realisasi'])[0,1]:+.3f}")
print(f"corr(residual, coverage) : "
      f"{np.corrcoef(terap['y_lgd_realisasi'] - pred, cov)[0,1]:+.3f}"
      "   <- porsi yang harus ditangkap lapisan agunan (tahap 2)")

Selisih R² dalam-domain vs luar-domain adalah **ukuran relevansi yang paling
langsung di seluruh proyek ini**. Sisanya — residual yang masih berkorelasi
dengan coverage — adalah bagian yang tidak mungkin dipelajari model satu tahap,
karena data latih SBA tidak punya kolom agunan sama sekali.

In [ ]:
# Topologi graf: ICIJ + AML. Relevansinya diukur dari bentuk sebaran derajat,
# bukan dari nilai absolutnya.
nodes = gold("gold_graph_nodes")
edges = gold("gold_graph_edges")
derajat = pd.concat([edges["src_node_id"], edges["dst_node_id"]]).value_counts()

fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.4))
ax[0].hist(np.log10(derajat.clip(lower=1)), bins=45, color="#8172B3")
ax[0].set_title("Sebaran derajat graf (log10)")
ax[0].set_xlabel("log10 derajat")

jenis = edges["jenis"].value_counts() if "jenis" in edges.columns else pd.Series(dtype=int)
if len(jenis):
    ax[1].barh(jenis.index[::-1].astype(str), jenis.values[::-1], color="#937860")
    ax[1].set_title("Jenis edge")
    ax[1].tick_params(labelsize=8)
plt.tight_layout(); plt.show()

print(f"simpul {len(nodes):,} | edge {len(edges):,}".replace(",", "."))
print(f"derajat: median {derajat.median():.0f}, p99 {derajat.quantile(.99):.0f}, "
      f"maks {derajat.max():.0f}")
print("\nEkor panjang berpangkat adalah bentuk yang MEMANG muncul di jaringan")
print("kepemilikan nyata - itulah sifat yang dipinjam dari ICIJ, bukan angkanya.")

## 6 · Ringkasan relevansi

Skor di bawah bukan hasil hitungan otomatis, melainkan pembacaan atas seluruh
bagian di atas. Kolom terakhir yang paling penting: **apa yang boleh diklaim**.

In [ ]:
RINGKAS = pd.DataFrame([
    dict(sumber="american_bankruptcy.csv",
         dipinjam="relasi rasio keuangan → gagal bayar",
         relevansi="SEDANG",
         boleh_diklaim="peringkat risiko relatif",
         tidak_boleh="level PD absolut untuk pencadangan"),
    dict(sumber="data.csv (Taiwan)",
         dipinjam="rasio tambahan, dicocokkan pada label",
         relevansi="RENDAH",
         boleh_diklaim="variasi rasio yang realistis",
         tidak_boleh="apa pun yang kausal — kolom tw_* sengaja dibuang dari ABT"),
    dict(sumber="SBAnational.csv",
         dipinjam="LGD, tenor, porsi penjaminan",
         relevansi="RENDAH–SEDANG",
         boleh_diklaim="rerata LGD portofolio",
         tidak_boleh="LGD per debitur tanpa lapisan agunan"),
    dict(sumber="corporate_rating.csv",
         dipinjam="sebaran rating per sektor",
         relevansi="SEDANG",
         boleh_diklaim="bentuk sebaran rating",
         tidak_boleh="pemetaan rating→PD"),
    dict(sumber="LI-Small_Trans.csv",
         dipinjam="topologi transfer",
         relevansi="TOPOLOGI SAJA",
         boleh_diklaim="pola siklus dan fan-in/out",
         tidak_boleh="nominal, kanal, atau base rate AML Indonesia"),
    dict(sumber="ICIJ Offshore Leaks",
         dipinjam="struktur kepemilikan & alamat bersama",
         relevansi="TOPOLOGI SAJA",
         boleh_diklaim="bentuk jaringan afiliasi",
         tidak_boleh="identitas, yurisdiksi, atau prevalensi di Indonesia"),
])
RINGKAS

## Kesimpulan

1. **Sinyalnya selamat melewati pipeline — bahkan terlalu selamat.** Arah relasi
   rasio→gagal bayar konsisten di 12 dari 13 rasio, dan struktur korelasi
   antar-rasio hampir utuh (korelasi antar matriks +0,98). Join sintetis di
   `joins.py` tidak merusak apa yang dipinjamnya. Tapi ia **memperkuatnya ~2×**
   (§3), karena pencocokan Taiwan memakai label sebagai bagian dari kuncinya.

2. **Ada dua bias yang menumpuk, dan keduanya searah:**

   | bias | asal | akibat |
   |---|---|---|
   | penguatan sinyal | pencocokan Taiwan berkunci label | AUC di sini > AUC sebenarnya |
   | kesenjangan populasi | emiten AS ≠ debitur komersial ID | relasi mungkin tidak berlaku sama sekali |

   Keduanya membuat model terlihat lebih baik daripada yang akan terjadi di
   portofolio nyata. Tidak ada yang saling menetralkan.

3. **Karena itu klaimnya harus dipisah.** Model ini boleh diklaim mampu
   *mengurutkan* risiko; ia tidak boleh diklaim menghasilkan *angka* PD atau LGD
   yang berlaku di portofolio nyata tanpa kalibrasi ulang di populasi tujuan.

4. **Bukti kuantitatif terkuat untuk laporan** ada di §5: selisih R² dalam-domain
   (0,331) vs luar-domain (0,197) adalah pengukuran langsung atas keterbatasan
   ini — sebuah angka, bukan pengakuan normatif.

---

### Satu kalimat untuk laporan

> Kualitas data menentukan apakah temuan kami nyata — dan §3–§4 menunjukkan
> sinyalnya memang selamat melewati pipeline. Relevansi data menentukan apakah
> temuan itu berlaku di portofolio BNI — dan §1 serta §5 menunjukkan batasnya
> secara terukur. Proyek ini membuktikan yang pertama dan menyatakan yang kedua;
> ia tidak berpura-pura menyelesaikan keduanya.